# Assignment 2 Orchestrator

This notebook runs the DWH2 workflow; students do **not** edit its Python cells.  
Prepare `group_NNN.csv` beside this notebook, and develop SQL in **DBeaver**.  
The group number comes from the roster filename; SQL questions are assigned automatically.

Expected layout:

```text
DWH2_NNN/
    AirQ_DWH2_NNN.ipynb
    group_NNN.csv
    csv/
    ddl/
    etl/                        # five supplied dimension ETLs; one fact ETL TODO
    sql/                        # assigned SQL files generated on first run
    sqldump/
    src/                        # supplied Python helpers
    environment-dwh.yaml
    a2_sql_assignments.json     # generated on first run
    a2_atoti_feasibility.csv    # generated on first run
```

The supplied DDL defines the complete DWH2 star, including the fact table. The five dimension ETL scripts are complete. Students implement **only the fact-table ETL** in `etl/a2_etl06_ft_param_city_month_location.sql` and their assigned SQL questions, all in external `.sql` files.


## Workflow

1. Complete both student rows in `group_NNN.csv`, and replace `NNN` in the folder and notebook names with your three-digit group number.
2. Run the notebook to generate your SQL question templates and Atoti feasibility CSV. Because the fact-table ETL is still a TODO, execution stops before any database reset. This is expected.
3. Implement **only the fact-table ETL** (`etl/a2_etl06_ft_param_city_month_location.sql`) in DBeaver. The five dimension ETLs are supplied. Rerun the notebook to populate the warehouse.
4. Solve your assigned SQL questions in DBeaver. Use the supplied Atoti web UI to assess feasibility and complete the Y/N CSV.
5. Rerun the notebook to review execution statuses, create the warehouse dump and see the run summary.

**Reuse the existing `airq` database and group account from Assignment 1.**  
Do not rerun the original bootstrap script: it can delete Assignment 1 work.  
The DWH2 notebook resets only the `stg2_NNN` and `dwh2_NNN` schemas.


## 1) External configuration — no notebook editing

Before running, complete **`group_NNN.csv`** next to this notebook:

- Keep exactly two rows, one for Student A and one for Student B. Enter the real names and student numbers in **both** rows, preserving student numbers as text.
- For a **one-student group**, enter the **same number and name** in A and B.
- For a **two-student group**, enter the two students' different numbers and names.
- Replace the sample names **Anton Chekhov** and **Agatha Christie**. Do not leave Student B blank.

`NNN` means the group's **three-digit number** (for example `007`). Replace it in the supplied folder, notebook and roster filenames. The notebook derives the group number from `group_NNN.csv`; no Python configuration is required.

The roster is validated **before** SQL files or database schemas are changed.  
After assignment, `a2_sql_assignments.json` locks the roster and assigned questions.  
**Do not edit or delete this manifest.** Existing student answers are preserved on reruns.


In [ ]:
from pathlib import Path
from IPython.display import HTML, Markdown, display
import pandas as pd

from src.dwh2_orchestrator import read_group_roster

ROOT_DIR = Path.cwd()
GROUP_INFO = read_group_roster(ROOT_DIR)
GroupNo = GROUP_INFO["group"]

# Exactly the same compact membership display as Assignment 1.
display(Markdown(f"### Group {GroupNo} — {GROUP_INFO['membership']}"))
display(HTML(pd.DataFrame(
    GROUP_INFO["members"],
    columns=["student_role", "student_number", "student_name"],
).to_html(index=False, escape=True)))


## 2) Setup, preflight and SQL assignment

Check the 15 supplied CSV files, two DDL files and six ETL scripts: **five completed dimension ETLs** and **one fact-table TODO**. Assign **four SQL1 and four SQL2 questions per student**: eight questions for a solo group, sixteen distinct questions for a pair.

The notebook creates comment-only SQL question files in `sql/` and `a2_atoti_feasibility.csv` when missing. Existing answers are **not overwritten**. The full question texts are in the assignment description.

The first run stops **before resetting database schemas** while the fact-table ETL is still a TODO.  
Implement that one SQL file in DBeaver, then rerun from the beginning.


In [ ]:
from src.dwh2_orchestrator import (
    STANDARD_CSV_ORDER,
    STAGING_DDL,
    WAREHOUSE_DDL,
    connect_database,
    create_sql_dump,
    database_user,
    load_csv_tables,
    preflight,
    pending_etl_files,
    print_run_summary,
    reset_schema,
    run_sql_file,
    run_sql_folder,
    schema_names,
)
from src.dwh2_sql import (
    assignment_table,
    check_feasibility,
    prepare_sql_workspace,
    run_assigned_sql,
)

VERBOSE_SQL = False  # internal logging setting; never a student input
SCHEMA_STG, SCHEMA_DWH = schema_names(GroupNo)
DB_USER = database_user(GroupNo)
CSV_DIR = ROOT_DIR / "csv"
DDL_DIR = ROOT_DIR / "ddl"
ETL_DIR = ROOT_DIR / "etl"
SQLDUMP_DIR = ROOT_DIR / "sqldump"
RUN_STATUS = {}

preflight_info = preflight(
    ROOT_DIR,
    group=GroupNo,
    standard_csv_order=STANDARD_CSV_ORDER,
)
RUN_STATUS["Preflight"] = "PASS"

# Both CSV roles have already passed roster validation. Allocate together.
people = {p["student_role"]: p for p in GROUP_INFO["members"]}
student_a = people["A"]["student_number"]
student_b = people["B"]["student_number"] if "B" in people else student_a
SQL_WORKSPACE = prepare_sql_workspace(
    ROOT_DIR,
    group=GroupNo,
    student_a=student_a,
    student_b=student_b,
)

assignment_style = (
    assignment_table(SQL_WORKSPACE)
    .style.hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("border-collapse", "collapse")]},
        {"selector": "th, td", "props": [
            ("border", "1px solid #aab4c0"),
            ("padding", "8px 16px"),
            ("text-align", "left"),
            ("white-space", "nowrap"),
        ]},
        {"selector": "th.col_heading", "props": [("font-weight", "bold")]},
    ])
)
display(assignment_style)
print(f"SQL templates: {len(SQL_WORKSPACE.created_sql)} new; "
      f"{len(SQL_WORKSPACE.preserved_sql)} existing files preserved.")
print(f"Feasibility CSV: {'created' if SQL_WORKSPACE.created_csv else 'preserved'}; "
      f"{SQL_WORKSPACE.added_csv_rows} new rows.")
RUN_STATUS["SQL1/SQL2 assignment"] = "PASS (fixed group allocation)"

# In the student starter, dimension ETLs 01-05 are complete and ETL 06 is a TODO.
# Generate SQL question templates first; block database resets for unfinished ETL.
ETL_PENDING = pending_etl_files(ETL_DIR)
if ETL_PENDING:
    raise RuntimeError(
        "Your SQL question templates and Atoti feasibility CSV are ready. "
        "The following ETL file(s) still contain only comments: "
        + ", ".join(ETL_PENDING) + ". "
        "Complete the fact-table ETL in DBeaver (the five dimension ETLs are "
        "provided), then rerun the notebook from the beginning. "
        "No PostgreSQL schemas were reset."
    )



## 3) Database connection

Connect to the existing `airq` database as `grp_NNN`. A password prompt is normal.  
**Do not rerun the database-dropping bootstrap from Assignment 1.**


In [ ]:
engine, DB_SETTINGS = connect_database(
    GroupNo,
    database="airq",
    host="localhost",
    port=5432,
)
RUN_STATUS["Database connection"] = "PASS"

## 4) Reset and create the staging schema

Reset only `stg2_NNN`, then execute the supplied `ddl/airq_create_stg2.sql`. The DWH1 schemas are not changed.


In [ ]:
reset_schema(engine, SCHEMA_STG, owner=DB_USER)
run_sql_file(
    DDL_DIR / STAGING_DDL,
    engine=engine,
    search_path=[SCHEMA_STG],
    progress=VERBOSE_SQL,
)
RUN_STATUS["Staging schema"] = "PASS"

## 5) Load CSV data into staging

Load the 15 source CSV tables in the fixed foreign-key-safe order.  
Unlike Assignment 1, no student-specified load-order CSV is required.


In [ ]:
csv_rows = load_csv_tables(
    CSV_DIR,
    STANDARD_CSV_ORDER,
    engine=engine,
    schema=SCHEMA_STG,
)

RUN_STATUS["CSV ingestion"] = (
    f"PASS ({len(csv_rows)} tables, {sum(csv_rows.values()):,} rows)"
)

## 6) Reset and create the dimensional warehouse

Reset only `dwh2_NNN`, then execute the supplied `ddl/airq_create_dwh2.sql`.  
It defines **five dimensions and one fact table** at Month x City x Parameter x Location Type grain.


In [ ]:
reset_schema(engine, SCHEMA_DWH, owner=DB_USER)
run_sql_file(
    DDL_DIR / WAREHOUSE_DDL,
    engine=engine,
    search_path=[SCHEMA_DWH],
    progress=VERBOSE_SQL,
)
RUN_STATUS["Warehouse schema"] = "PASS"

## 7) SQL-first ETL

The notebook executes all six `etl/a2_etl*.sql` files in filename order.  
**ETL 01-05 (the five dimensions) are supplied complete and are not student implementation tasks.**  
You implement and test **only ETL 06** (`a2_etl06_ft_param_city_month_location.sql`) in DBeaver.

Your fact ETL must populate the supplied fact table at **Month x City x Parameter x Location Type** grain, resolving its dimension keys and calculating its measures. The notebook supplies the staging and DWH schemas through the SQL search path; do not hardcode schema names.


In [ ]:
etl_results = run_sql_folder(
    ETL_DIR,
    "a2_etl*.sql",
    engine=engine,
    search_path=[SCHEMA_DWH, SCHEMA_STG],
    progress=VERBOSE_SQL,
)
RUN_STATUS["ETL"] = f"PASS ({len(etl_results)} SQL files)"


## 8) SQL1 and SQL2 — execution status

The assigned questions are in `sql/` (SQL1: 101-116; SQL2: 201-216). Use the assignment description for the full questions. Develop **one read-only `SELECT` or `WITH ... SELECT` per file** in DBeaver, using plain warehouse table names.

The tables below show `TEMPLATE`, `EXECUTED` or `ERROR`, returned row counts and short diagnostics **separately for Student A and Student B**. Actual SQL result rows are inspected in DBeaver, not displayed here. **`EXECUTED` does not mean the answer is correct.**


In [ ]:
# Show status and row counts only; students inspect SQL result rows in DBeaver.
sql_report = run_assigned_sql(
    engine,
    SQL_WORKSPACE,
    warehouse_schema=SCHEMA_DWH,
)
for student in SQL_WORKSPACE.assignments:
    student_rows = sql_report.loc[
        sql_report["Student"] == student,
        ["Question", "File", "Status", "Rows", "Detail"],
    ]
    display(Markdown(f"### Student {student} - SQL execution"))
    display(HTML(student_rows.to_html(
        index=False, escape=True, na_rep="-",
        formatters={"Rows": lambda n: str(int(n)) if pd.notna(n) else "-"},
    )))
executed = int((sql_report["Status"] == "EXECUTED").sum())
total = len(sql_report)
RUN_STATUS["SQL execution"] = (
    f"PASS ({executed}/{total} executed; correctness not checked)"
    if executed == total else f"INCOMPLETE ({executed}/{total} executed)"
)

## 9) Atoti feasibility CSV

The notebook creates one group-level `a2_atoti_feasibility.csv` with columns `student`, `question_id`, `atoti_ui_possible`, `reason`.

For each assigned question, enter **`Y`** if the required business values can be obtained with the supplied Atoti pivot UI and predefined measures; leave `reason` empty. Otherwise enter **`N`** and give a short explanation in `reason`.

Do not change student labels or question IDs. Existing answers are preserved. The tables below check **completion and format, not correctness**, separately for Student A and Student B. Do not create custom cube measures or write MDX for this exercise.


In [ ]:
feasibility_report = check_feasibility(SQL_WORKSPACE)
for student in SQL_WORKSPACE.assignments:
    student_rows = feasibility_report.loc[
        feasibility_report["Student"] == student,
        ["Question", "Y/N", "Status", "Detail"],
    ]
    display(Markdown(f"### Student {student} - Atoti feasibility"))
    display(HTML(student_rows.to_html(index=False, escape=True)))
ready = int((feasibility_report["Status"] == "READY").sum())
total = len(feasibility_report)
RUN_STATUS["Atoti Y/N feasibility"] = (
    f"PASS ({ready}/{total} filled; correctness not checked)"
    if ready == total else f"INCOMPLETE ({ready}/{total} filled)"
)

## 10) Supplied Atoti 6.2.2 cube

Load the provided Atoti cube from the populated warehouse: **one fact and five dimensions**, with predefined hierarchies and measures. Students do not edit Python cube code or write MDX. Open the link below and create a **Pivot Table** to explore the assigned feasibility questions.

Use the existing `dwh` Conda environment. Keep Jupyter running while using Atoti. After rebuilding the warehouse, restart the kernel and rerun the notebook to refresh the in-memory cube.

**Interpretation:** Recorded-value averages should be compared for one pollutant at a time. Sums of monitoring-day counts across different pollutants or locations are not necessarily counts of distinct calendar dates.


In [ ]:
from src.dwh2_atoti import (
    ATOTI_VERSION,
    VISIBLE_MEASURES,
    build_atoti_cube,
    check_atoti_environment,
    validate_atoti_totals,
)
import sys

# Verify which Python executable the Jupyter kernel really uses, then check
# the Atoti package and its client/core/server components BEFORE starting Java.
print("Jupyter Python:", sys.executable)
ATOTI_COMPONENT_VERSIONS = check_atoti_environment()
print("Atoti version:", ATOTI_VERSION)
for package, installed in ATOTI_COMPONENT_VERSIONS.items():
    print(f"  {package}: {installed}")

# Build the in-memory Atoti cube from the already populated DWH2 schema.
# Restart the kernel before rebuilding to avoid duplicate running servers.
session, cube, ATOTI_EXPECTED = build_atoti_cube(engine, SCHEMA_DWH)


### Validate the cube against PostgreSQL

The four additive totals must match the populated warehouse.  
This checks the cube's data loading, **not** whether the student's ETL is correct.


In [ ]:
atoti_checks = validate_atoti_totals(cube, ATOTI_EXPECTED)
display(atoti_checks)
RUN_STATUS["Atoti cube"] = f"PASS ({ATOTI_EXPECTED['Fact rows']:,} fact rows; 4/4 totals)"


### Open the Atoti web interface

Open the printed URL or notebook link and create a **Pivot Table**. Use the supplied time, country/city, parameter, location and alert hierarchies and existing measures. Filtering and rearranging pivot tables is allowed; do not create custom measures or write MDX.


In [ ]:
print("Atoti browser URL:", session.url)
print("Visible measures:", len(VISIBLE_MEASURES))
# In JupyterLab the Atoti link uses the Jupyter proxy if available.
session.link


## 11) Create the warehouse SQL dump

Export only `dwh2_NNN` using `pg_dump` with `--no-owner --no-privileges`, as in Assignment 1.  
PostgreSQL command-line tools must be installed.


In [ ]:
dump_path = create_sql_dump(
    SQLDUMP_DIR,
    group=GroupNo,
    db_settings=DB_SETTINGS,
    schema=SCHEMA_DWH,
)
RUN_STATUS["SQL dump"] = f"PASS ({dump_path.name})"

## 12) Run summary

In [ ]:
print_run_summary(RUN_STATUS, group=GroupNo)